# UPDATE METADATA
## Inizializzazione ed Import

In [2]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [127]:
level = '1'

In [128]:
file_codes = ['ADNI_DIAN_COMPARISON']
#'ADNIMERGE', 'UCSFFSX', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 
# 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL'] #, 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 
# 'BLCHANGE', 'DXSUM', 

In [129]:
search = client.query_files(
    query={'custom.level' : 'cleaned_0'+level, 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [130]:
print(zip_files.keys())

dict_keys(['ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_01.csv'])


In [131]:
df = zip_files['ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_01.csv']
df_ = df.copy(deep=True)

In [132]:
df_unknown = df_[df_['METHOD']=='unknown'].copy(deep=True)

In [133]:
df_unknown.columns

Index(['COHORT', 'RID', 'VISCODE', 'VISIT_MONTH', 'EXAMDATE', 'AGE', 'CDRGLOB',
       'CSF_DATE', 'MRI_SCANDATE', 'ICV', 'Hippocampus', 'ETHNICITY', 'RACE',
       'MARRY', 'CDRSB', 'GENDER', 'AB42_CSF', 'PT181_CSF', 'TTAU_CSF',
       'AB40_CSF', 'AB4240_CSF', 'PET_SCANDATE', 'APOE', 'APOE_4',
       'DIAN_MUTATION', 'MMSE', 'METHOD', 'TTAU_AB42_CSF', 'PT181_AB42_CSF',
       'Apositive', 'Tpositive', 'Npositive'],
      dtype='object')

{'AB42_CSF': np.False_, 'PT181_CSF': np.False_, 'TTAU_CSF': np.False_, 'AB40_CSF': np.False_, 'AB4240_CSF': np.False_, 'TTAU_AB42_CSF': np.False_, 'PT181_AB42_CSF': np.False_, 'Apositive': np.False_, 'Tpositive': np.False_, 'Npositive': np.False_, 'RID': np.True_}


# Import support file already populated

In [55]:
support_file = pd.read_excel('ADNI_variables_cleaned'+ level +'.xlsx')
dataCleaner = DataCleaner(support_file=support_file)

# Get & update metadata

In [56]:
for file_name in zip_files.keys():
    df = zip_files[file_name]
    final_df = df.copy(deep=True)
    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(final_df, new_level='cleaned_0'+level, file_name=file_name, prefix='cleaned/single_file', updated_support_file=support_file)  

    metadata = client.get_metadata(object_name = 'cleaned/single_file/' + file_name)
    metadata_id = metadata['metadata']['_id']

    result = client.update_file(
        object_name= 'cleaned/single_file/' + file_name,
        metadata=updated_metadata
    )

    print('\n###  ', file_name, '  ##################\n', updated_metadata)


###   APOERES_11Aug2025_03.csv   ##################
 {'cofattori': ['APOE'], 'file_code': 'APOERES', 'level': 'cleaned_03', 'norm_intervallo': [], 'norm_scala': [], 'norm_scale_value': [], 'norm_volume': [], 'population': ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3'], 'predittori': [], 'source': 'ADNI', 'volume_norm_values': []}


# Verify uploaded metadata

In [116]:
level = '4'
file_codes = ['ADSP_PHC_BIOMARKER', 'ADNI_DIAN_COMPARISON', 'APOERES', 'UPENNBIOMK_ADNIDIAN_ES_2017', 'UPENNBIOMK_ROCHE_ELECSYS', 'EUROIMMUN', 'FUJIREBIOABETA', 'SALADAX_BIOMEDICAL', 'MESOSCALE', 'UPENNBIOMK_MASTER', 'UPENN_2DUPLC_CRM']
search = client.query_files(
    query={'custom.level' : 'cleaned_0'+level, 'custom.source' : 'ADNI', 'custom.file_code': file_codes})


In [117]:
obj_names = [x['object_name'] for x in search['included_files']]
print(obj_names)

['cleaned/single_file/APOERES_11Aug2025_04.csv', 'cleaned/single_file/ADSP_PHC_BIOMARKER_25Jul2025_04.csv', 'cleaned/single_file/UPENNBIOMK_ROCHE_ELECSYS_09Oct2025_04.csv', 'cleaned/single_file/UPENNBIOMKADNIDIAN2017_09Oct2025_04.csv', 'cleaned/single_file/ADNI_EUROIMMUN_11Aug2025_04.csv', 'cleaned/single_file/FUJIREBIOABETA_11Aug2025_04.csv', 'cleaned/single_file/SALADAX_BIOMEDICAL_11Aug2025_04.csv', 'cleaned/single_file/ADNI-DIAN_Comparison_Study_Data_Subset_05_23_22_23Oct2025_04.csv', 'cleaned/single_file/ADNI_MESOSCALE_23Oct2025_04.csv', 'cleaned/single_file/UPENNBIOMK_MASTER_23Oct2025_04.csv', 'cleaned/single_file/UPENNMSMSABETA2CRM_23Oct2025_04.csv']


In [118]:
i = 0

In [126]:
metadata = client.get_metadata(object_name=obj_names[i])
display(metadata['metadata']['custom'])
i += 1
print(i)

{'cofattori': ['APOE',
  'APOE_4',
  'GENDER/female',
  'GENDER/male',
  'MARRY/divorced',
  'MARRY/married',
  'MARRY/single',
  'MARRY/widowed',
  'ETHNICITY/latino',
  'ETHNICITY/not_latino',
  'RACE/Asian',
  'RACE/Black',
  'RACE/Native_american',
  'RACE/White'],
 'cofattori_metadata': {'APOE_4': [0, 2, 'increasing']},
 'file_code': 'ADNI_DIAN_COMPARISON',
 'level': 'cleaned_04',
 'norm_intervallo': ['ICV',
  'Hippocampus',
  'AB42_CSF',
  'PT181_CSF',
  'TTAU_CSF',
  'AB40_CSF',
  'AB4240_CSF',
  'TTAU_AB42_CSF',
  'PT181_AB42_CSF'],
 'norm_scala': ['CDRSB', 'CDRGLOB', 'MMSE'],
 'norm_scale_value': {'AB40_CSF': {'elecsys': [6333.2, 33907.4, 'inverse'],
   'massospectrometry': [3483.5, 15231.82, 'inverse'],
   'unknown': [4220.82, 28595.399999999994, 'inverse']},
  'AB4240_CSF': {'elecsys': [0.0209, 0.1157, 'inverse'],
   'massospectrometry': [0.0509, 0.2309, 'inverse'],
   'unknown': [0.022208294568095574, 0.22065756429474834, 'inverse']},
  'AB42_CSF': {'elecsys': [272.106, 292

8
